# Do Institutions Cause Prosperity? An IV Tutorial in Python

**Tutorial companion:** [carlos-mendez.org/post/python_iv/](https://carlos-mendez.org/post/python_iv/)

This notebook replicates Acemoglu, Johnson and Robinson (2001) — **AJR** — using a hybrid Python stack: [`pyfixest`](https://pyfixest.org/) for 2SLS estimates and OLS comparisons, and [`linearmodels`](https://bashtage.github.io/linearmodels/) for Kleibergen-Paap F, Hansen J, and Wu-Hausman tests.

AJR proposed using the **mortality rate of European settlers** during colonization as an *instrumental variable* for modern institutional quality. Places where Europeans died en masse became *extractive* colonies; places where they survived became *settler* colonies with European-style property-rights protections.

The case study question: **"Do better institutions cause higher GDP per capita, or are they merely correlated with it?"**

## Learning objectives

- **Recognize** when OLS is biased by reverse causality, omitted variables, and measurement error.
- **State** the three IV conditions: relevance, exclusion, and exogeneity.
- **Estimate** the AJR (2001) 2SLS coefficient using `pyfixest.feols` and `linearmodels.iv.IV2SLS`.
- **Diagnose** weak instruments using the Kleibergen-Paap F-statistic and Stock-Yogo critical values.
- **Interpret** the 2SLS coefficient as a Local Average Treatment Effect (LATE).
- **Test** the exclusion restriction with the Hansen J overidentification test.

## Setup and dependencies

In [ ]:
!pip install pyfixest linearmodels -q

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfixest as pf
from linearmodels.iv import IV2SLS

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.options.display.float_format = "{:.4f}".format

# Site color palette (dark theme)
STEEL_BLUE  = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK  = "#141413"
TEAL        = "#00d4c8"
DARK_NAVY   = "#0f1729"
GRID_LINE   = "#1f2b5e"
LIGHT_TEXT  = "#c8d0e0"
WHITE_TEXT  = "#e8ecf2"

plt.rcParams.update({
    "figure.facecolor": DARK_NAVY,
    "axes.facecolor":   DARK_NAVY,
    "axes.edgecolor":   DARK_NAVY,
    "axes.linewidth":   0,
    "axes.labelcolor":  LIGHT_TEXT,
    "axes.titlecolor":  WHITE_TEXT,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.left":   False,
    "axes.spines.bottom": False,
    "axes.grid":        True,
    "grid.color":       GRID_LINE,
    "grid.linewidth":   0.6,
    "grid.alpha":       0.8,
    "xtick.color":      LIGHT_TEXT,
    "ytick.color":      LIGHT_TEXT,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "text.color":       WHITE_TEXT,
    "font.size":        11,
    "legend.frameon":   False,
    "legend.fontsize":  10,
    "legend.labelcolor": LIGHT_TEXT,
    "savefig.facecolor": DARK_NAVY,
    "savefig.edgecolor": DARK_NAVY,
})

# Data source: GitHub raw URL for one-click replicability
DATA_URL = "https://raw.githubusercontent.com/cmg777/starter-academic-v501/master/content/post/stata_iv"

def load_dta(table_n):
    """Load AJR's maketable{N}.dta from the GitHub repo."""
    return pd.read_stata(f"{DATA_URL}/maketable{table_n}.dta")

def banner(text):
    print("\n" + "=" * 64)
    print(f"  {text}")
    print("=" * 64)

def coef_se(model, var):
    return model.coef()[var], model.se()[var]

def stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

def fmt_with_stars(beta, p):
    return f"{beta:.3f}{stars(p)}"

def vars_in(formula):
    out = []
    for piece in formula.replace("~", " ").replace("|", " ").replace("+", " ").split():
        s = piece.strip()
        if s and s != "1" and s.isidentifier():
            out.append(s)
    return list(dict.fromkeys(out))

print(f"Data source: {DATA_URL}")

## Data overview (Table 1)

AJR provide eight datasets — one per table. Table 1 covers the full ~163-country world; Tables 2-8 narrow to the 64-country **base sample** of ex-colonies with valid settler-mortality data.

In [ ]:
df1 = load_dta(1)

print("*** Whole world ***")
print(df1[["logpgp95", "avexpr", "euro1900"]].describe().T)

print("\n*** AJR base sample (baseco==1) ***")
base1 = df1[df1["baseco"] == 1]
print(base1[["logpgp95", "avexpr", "euro1900", "logem4"]].describe().T)

The base sample has 64 former colonies. Restricting to ex-colonies lowers mean `avexpr` from 6.99 to 6.52 (weaker institutions) and mean `euro1900` from 30.1 to 16.2 (fewer European settlers). The instrument `logem4` ranges from 2.15 to 7.99 — nearly six log points of variation.

## The naive OLS benchmark (Table 2)

Before instrumenting, we estimate OLS to see what the naive slope looks like. All standard errors are heteroskedasticity-robust (HC1).

In [ ]:
df2 = load_dta(2)

m2_specs = [
    ("Col 1: whole world",         "logpgp95 ~ avexpr",                                  None),
    ("Col 2: base sample",         "logpgp95 ~ avexpr",                                  df2["baseco"] == 1),
    ("Col 3: + latitude",          "logpgp95 ~ avexpr + lat_abst",                       None),
    ("Col 4: + lat + continents",  "logpgp95 ~ avexpr + lat_abst + africa + asia + other", None),
    ("Col 5: base + latitude",     "logpgp95 ~ avexpr + lat_abst",                       df2["baseco"] == 1),
    ("Col 6: base + lat + cont.",  "logpgp95 ~ avexpr + lat_abst + africa + asia + other", df2["baseco"] == 1),
    ("Col 7: loghjypl, world",     "loghjypl ~ avexpr",                                  None),
    ("Col 8: loghjypl, base",      "loghjypl ~ avexpr",                                  df2["baseco"] == 1),
]

for name, fml, mask in m2_specs:
    sub = df2 if mask is None else df2[mask]
    sub = sub.dropna(subset=[c for c in vars_in(fml) if c in sub.columns])
    m = pf.feols(fml, data=sub, vcov="HC1")
    b, se = coef_se(m, "avexpr")
    p = m.pvalue()["avexpr"]
    print(f"{name:30s}  avexpr = {fmt_with_stars(b, p):>10s}  (SE {se:.3f})  N={int(m._N)}")

The OLS coefficient is stable: 0.532 (full sample), 0.522 (base sample), down to 0.390 with continent dummies. But these estimates carry three known biases: reverse causality, omitted variables, and measurement error. We need IV to disentangle.

## Determinants of institutions (Table 3)

Table 3 previews the first stage: settler mortality shapes settlement, settlement shapes early institutions, and early institutions persist into modern institutions.

In [ ]:
df3 = load_dta(3)
df3 = df3[(df3["excolony"] == 1) & df3["extmort4"].notna()].copy()
df3["euro1900"] = df3["euro1900"] / 100.0

# Panel A: DV = avexpr (current institutions)
panel_a_specs = [
    ("A.c1",  "avexpr ~ cons00a"),
    ("A.c2",  "avexpr ~ cons00a + lat_abst"),
    ("A.c3",  "avexpr ~ democ00a"),
    ("A.c4",  "avexpr ~ democ00a + lat_abst"),
    ("A.c5",  "avexpr ~ cons1 + indtime"),
    ("A.c6",  "avexpr ~ cons1 + indtime + lat_abst"),
    ("A.c7",  "avexpr ~ euro1900"),
    ("A.c8",  "avexpr ~ euro1900 + lat_abst"),
    ("A.c9",  "avexpr ~ logem4"),
    ("A.c10", "avexpr ~ logem4 + lat_abst"),
]

print("*** Panel A: DV = current expropriation protection (avexpr) ***")
for name, fml in panel_a_specs:
    sub = df3.dropna(subset=[c for c in vars_in(fml) if c in df3.columns])
    sub = sub[sub["logpgp95"].notna()]
    m = pf.feols(fml, data=sub, vcov="HC1")
    rhs = fml.split("~")[1].strip().split("+")[0].strip()
    b, se = coef_se(m, rhs)
    p = m.pvalue()[rhs]
    print(f"  {name:6s}  {rhs:12s} = {fmt_with_stars(b, p):>10s}  (SE {se:.3f})  N={int(m._N)}")

In [ ]:
# Panel B: DV = early institution
panel_b_specs = [
    ("B.c1",  "cons00a ~ euro1900"),
    ("B.c2",  "cons00a ~ euro1900 + lat_abst"),
    ("B.c3",  "cons00a ~ logem4"),
    ("B.c4",  "cons00a ~ logem4 + lat_abst"),
    ("B.c5",  "democ00a ~ euro1900"),
    ("B.c6",  "democ00a ~ euro1900 + lat_abst"),
    ("B.c7",  "democ00a ~ logem4"),
    ("B.c8",  "democ00a ~ logem4 + lat_abst"),
    ("B.c9",  "euro1900 ~ logem4"),
    ("B.c10", "euro1900 ~ logem4 + lat_abst"),
]

print("*** Panel B: DV = early institutions (cons00a/democ00a/euro1900) ***")
for name, fml in panel_b_specs:
    sub = df3.dropna(subset=[c for c in vars_in(fml) if c in df3.columns])
    sub = sub[sub["logpgp95"].notna()]
    m = pf.feols(fml, data=sub, vcov="HC1")
    rhs = fml.split("~")[1].strip().split("+")[0].strip()
    b, se = coef_se(m, rhs)
    p = m.pvalue()[rhs]
    print(f"  {name:6s}  {rhs:12s} = {fmt_with_stars(b, p):>10s}  (SE {se:.3f})  N={int(m._N)}")

## First stage, reduced form, and Figures 1-2

An instrument must be **relevant** — it must move the endogenous regressor. We test this with the first-stage regression and visualize the relationship.

In [ ]:
df4 = load_dta(4)
base = df4[df4["baseco"] == 1].dropna(subset=["logpgp95", "avexpr", "logem4"]).copy()

# First-stage F via linearmodels (canonical KP-F + DWH)
y = base["logpgp95"].values
X_endog = base[["avexpr"]]
X_exog = pd.DataFrame({"const": np.ones(len(base))}, index=base.index)
Z = base[["logem4"]]
res4 = IV2SLS(y, X_exog, X_endog, Z).fit(cov_type="robust")

fs_F = float(res4.first_stage.diagnostics.loc["avexpr", "f.stat"])
fs_pval = float(res4.first_stage.diagnostics.loc["avexpr", "f.pval"])
print(f"First-stage robust F (~Kleibergen-Paap):  {fs_F:.2f}  (p = {fs_pval:.2e})")
print(f"Stock-Yogo 10% maximal IV size threshold:  16.38 (IID)")

dwh = res4.wu_hausman()
print(f"Wu-Hausman endogeneity F = {dwh.stat:.3f}, p = {dwh.pval:.4f}")

In [ ]:
# Helper for country-label annotations
def annotate_scatter(ax, x, y, labels):
    for xi, yi, lab in zip(x, y, labels):
        ax.annotate(lab, (xi, yi), textcoords="offset points", xytext=(4, 2),
                    fontsize=6, color=TEAL, alpha=0.8)

# Figure 1: First stage (logem4 -> avexpr)
fig1, ax1 = plt.subplots(figsize=(10, 6.5))
ax1.scatter(base["logem4"], base["avexpr"], color=STEEL_BLUE, s=28, alpha=0.85, edgecolor="none")
annotate_scatter(ax1, base["logem4"], base["avexpr"], base["shortnam"])
xfit = np.linspace(base["logem4"].min(), base["logem4"].max(), 100)
fs_slope = res4.first_stage.individual["avexpr"].params["logem4"]
fs_intercept = res4.first_stage.individual["avexpr"].params["const"]
ax1.plot(xfit, fs_intercept + fs_slope * xfit, color=WARM_ORANGE, linewidth=2.2)
ax1.set_title("Figure 1. First stage: settler mortality predicts institutions",
              fontsize=13, color=WHITE_TEXT, pad=22)
ax1.text(0.5, 1.02, "Base sample of 64 ex-colonies (AJR 2001 Table 4)",
         transform=ax1.transAxes, ha="center", fontsize=10, color=LIGHT_TEXT)
ax1.set_xlabel("Log settler mortality (logem4)")
ax1.set_ylabel("Avg. protection from expropriation, 1985-95 (avexpr)")
plt.tight_layout()
plt.show()

The negative slope is unmistakable. Australia, New Zealand, and the United States (lowest mortality) sit at `avexpr` ~ 9-10. Sierra Leone, Niger, and Mali (highest mortality) sit near 3.5-5. The fit captures 27% of variation in modern institutions.

In [ ]:
# Figure 2: Reduced form (logem4 -> logpgp95)
m_rf = pf.feols("logpgp95 ~ logem4", data=base, vcov="HC1")
rf_slope = m_rf.coef()["logem4"]
rf_intercept = m_rf.coef()["Intercept"]

fig2, ax2 = plt.subplots(figsize=(10, 6.5))
ax2.scatter(base["logem4"], base["logpgp95"], color=STEEL_BLUE, s=28, alpha=0.85, edgecolor="none")
annotate_scatter(ax2, base["logem4"], base["logpgp95"], base["shortnam"])
ax2.plot(xfit, rf_intercept + rf_slope * xfit, color=WARM_ORANGE, linewidth=2.2)
ax2.set_title("Figure 2. Reduced form: settler mortality predicts log GDP",
              fontsize=13, color=WHITE_TEXT, pad=22)
ax2.text(0.5, 1.02, "Base sample of 64 ex-colonies (AJR 2001 Table 4)",
         transform=ax2.transAxes, ha="center", fontsize=10, color=LIGHT_TEXT)
ax2.set_xlabel("Log settler mortality (logem4)")
ax2.set_ylabel("Log GDP per capita, PPP, 1995 (logpgp95)")
plt.tight_layout()
plt.show()

print(f"\nFirst-stage slope (logem4 -> avexpr):   {fs_slope:.3f}")
print(f"Reduced-form slope (logem4 -> logpgp95): {rf_slope:.3f}")
print(f"Implied 2SLS beta = RF / FS = {rf_slope:.3f} / {fs_slope:.3f} = {rf_slope/fs_slope:.3f}")

The IV decomposes the reduced-form effect into two pieces. Dividing the reduced-form slope by the first-stage slope gives: **-0.573 / -0.607 = 0.944** — exactly the 2SLS coefficient.

## The main 2SLS estimate (Table 4)

This is the headline result. We instrument `avexpr` with `logem4`, with HC-robust standard errors.

In [ ]:
def iv_with_kpf(df_in, formula):
    sub = df_in.dropna(subset=[c for c in vars_in(formula) if c in df_in.columns]).copy()
    m_pf = pf.feols(formula, data=sub, vcov="HC1")
    return m_pf, sub

def run_iv2sls(sub, dep, exog_cols, endog_cols, inst_cols):
    sub = sub.dropna(subset=[dep] + exog_cols + endog_cols + inst_cols).copy()
    y_ = sub[dep].values
    if exog_cols:
        X_exog_ = sub[exog_cols].copy()
        X_exog_["const"] = 1.0
    else:
        X_exog_ = pd.DataFrame({"const": np.ones(len(sub))}, index=sub.index)
    res = IV2SLS(y_, X_exog_, sub[endog_cols], sub[inst_cols]).fit(cov_type="robust")
    fs_diag = res.first_stage.diagnostics
    kpf = float(fs_diag.loc[endog_cols[0], "f.stat"]) if endog_cols[0] in fs_diag.index else np.nan
    return res, kpf

In [ ]:
df4 = load_dta(4)
df4 = df4[df4["baseco"] == 1].copy()
df4["other_cont"] = ((df4["shortnam"] == "AUS") | (df4["shortnam"] == "MLT") |
                    (df4["shortnam"] == "NZL")).astype(int)

# Panel B: 2SLS (IV with logem4)
table4_specs = [
    ("c1: base",                  "logpgp95 ~ 1 | avexpr ~ logem4",                              None,                    "logpgp95", [],                                ["avexpr"], ["logem4"]),
    ("c2: + lat",                 "logpgp95 ~ lat_abst | avexpr ~ logem4",                       None,                    "logpgp95", ["lat_abst"],                      ["avexpr"], ["logem4"]),
    ("c3: -Neo-Europes",          "logpgp95 ~ 1 | avexpr ~ logem4",                              df4["rich4"] != 1,       "logpgp95", [],                                ["avexpr"], ["logem4"]),
    ("c4: -Neo-Europes + lat",    "logpgp95 ~ lat_abst | avexpr ~ logem4",                       df4["rich4"] != 1,       "logpgp95", ["lat_abst"],                      ["avexpr"], ["logem4"]),
    ("c5: -Africa",               "logpgp95 ~ 1 | avexpr ~ logem4",                              df4["africa"] != 1,      "logpgp95", [],                                ["avexpr"], ["logem4"]),
    ("c6: -Africa + lat",         "logpgp95 ~ lat_abst | avexpr ~ logem4",                       df4["africa"] != 1,      "logpgp95", ["lat_abst"],                      ["avexpr"], ["logem4"]),
    ("c7: + continents",          "logpgp95 ~ africa + asia + other_cont | avexpr ~ logem4",     None,                    "logpgp95", ["africa", "asia", "other_cont"], ["avexpr"], ["logem4"]),
    ("c8: + continents + lat",    "logpgp95 ~ africa + asia + other_cont + lat_abst | avexpr ~ logem4", None,             "logpgp95", ["africa", "asia", "other_cont", "lat_abst"], ["avexpr"], ["logem4"]),
    ("c9: loghjypl",              "loghjypl ~ 1 | avexpr ~ logem4",                              None,                    "loghjypl", [],                                ["avexpr"], ["logem4"]),
]

print("*** Panel B: 2SLS (IV with logem4) ***\n")
for tag, fml, mask, dep, exog, endog, inst in table4_specs:
    sub = df4 if mask is None else df4[mask]
    m_pf, sub2 = iv_with_kpf(sub, fml)
    b_pf, se_pf = coef_se(m_pf, "avexpr")
    p_pf = m_pf.pvalue()["avexpr"]
    res_lm, kpf = run_iv2sls(sub2, dep, exog, endog, inst)
    b_lm = res_lm.params["avexpr"]
    se_lm = res_lm.std_errors["avexpr"]
    print(f"  IV {tag:30s} pyfixest b={b_pf:.3f} (SE {se_pf:.3f})  "
          f"LM b={b_lm:.3f} (SE {se_lm:.3f})  KP-F={kpf:.2f}  N={int(m_pf._N)}")

In [ ]:
# Tab 4 Col 1 detailed diagnostics
print("*** Tab 4 Col 1 -- full diagnostics ***")
res_c1, kpf_c1 = run_iv2sls(df4, "logpgp95", [], ["avexpr"], ["logem4"])
dwh = res_c1.wu_hausman()
print(f"  IV beta = {res_c1.params['avexpr']:.4f}  (SE {res_c1.std_errors['avexpr']:.4f})")
print(f"  CI 95% = [{res_c1.conf_int().loc['avexpr', 'lower']:.3f}, "
      f"{res_c1.conf_int().loc['avexpr', 'upper']:.3f}]")
print(f"  First-stage robust F (~KP-F): {kpf_c1:.3f}")
print(f"  Wu-Hausman endogeneity F = {dwh.stat:.3f}, p = {dwh.pval:.4f}")

In [ ]:
# Panel C: paired OLS comparisons
print("*** Panel C: OLS comparisons ***\n")
table4_ols_specs = [
    ("c10: base",          "logpgp95 ~ avexpr",                                   None),
    ("c11: + lat",         "logpgp95 ~ avexpr + lat_abst",                        None),
    ("c12: -NeoEur",       "logpgp95 ~ avexpr",                                   df4["rich4"] != 1),
    ("c13: -NeoEur + lat", "logpgp95 ~ avexpr + lat_abst",                        df4["rich4"] != 1),
    ("c14: -Africa",       "logpgp95 ~ avexpr",                                   df4["africa"] != 1),
    ("c15: -Africa + lat", "logpgp95 ~ avexpr + lat_abst",                        df4["africa"] != 1),
    ("c16: + continents",  "logpgp95 ~ avexpr + africa + asia + other_cont",      None),
    ("c17: + cont + lat",  "logpgp95 ~ avexpr + lat_abst + africa + asia + other_cont", None),
    ("c18: loghjypl",      "loghjypl ~ avexpr",                                   None),
]

for tag, fml, mask in table4_ols_specs:
    sub = df4 if mask is None else df4[mask]
    sub = sub.dropna(subset=[c for c in vars_in(fml) if c in sub.columns])
    m = pf.feols(fml, data=sub, vcov="HC1")
    b, se = coef_se(m, "avexpr")
    p = m.pvalue()["avexpr"]
    print(f"  OLS {tag:30s} b={b:.3f} (SE {se:.3f})  N={int(m._N)}")

The 2SLS coefficient on `avexpr` is **0.944** — **81% larger** than OLS (0.522). The Wu-Hausman test rejects OLS consistency (F = 24.22, p < 0.0001). The IV > OLS gap suggests measurement error dominates: institutional quality is a noisy proxy, and de-noising via IV reveals a steeper causal slope.

## Robustness 1: colonial, legal, and religious controls (Table 5)

In [ ]:
df5 = load_dta(5)
df5 = df5[df5["baseco"] == 1].copy()

table5_specs = [
    ("c1: + Brit/French",          "logpgp95 ~ f_brit + f_french | avexpr ~ logem4",                       None,                  ["f_brit", "f_french"]),
    ("c2: + Brit/French + lat",    "logpgp95 ~ f_brit + f_french + lat_abst | avexpr ~ logem4",            None,                  ["f_brit", "f_french", "lat_abst"]),
    ("c3: British only",           "logpgp95 ~ 1 | avexpr ~ logem4",                                       df5["f_brit"] == 1,    []),
    ("c4: British only + lat",     "logpgp95 ~ lat_abst | avexpr ~ logem4",                                df5["f_brit"] == 1,    ["lat_abst"]),
    ("c5: + French legal",         "logpgp95 ~ sjlofr | avexpr ~ logem4",                                  None,                  ["sjlofr"]),
    ("c6: + French legal + lat",   "logpgp95 ~ sjlofr + lat_abst | avexpr ~ logem4",                       None,                  ["sjlofr", "lat_abst"]),
    ("c7: + religion",             "logpgp95 ~ catho80 + muslim80 + no_cpm80 | avexpr ~ logem4",           None,                  ["catho80", "muslim80", "no_cpm80"]),
    ("c8: + religion + lat",       "logpgp95 ~ catho80 + muslim80 + no_cpm80 + lat_abst | avexpr ~ logem4", None,                 ["catho80", "muslim80", "no_cpm80", "lat_abst"]),
    ("c9: kitchen sink",           "logpgp95 ~ f_french + sjlofr + catho80 + muslim80 + no_cpm80 + lat_abst | avexpr ~ logem4", None, ["f_french", "sjlofr", "catho80", "muslim80", "no_cpm80", "lat_abst"]),
]

for tag, fml, mask, exog in table5_specs:
    sub = df5 if mask is None else df5[mask]
    m_pf, sub2 = iv_with_kpf(sub, fml)
    res, kpf = run_iv2sls(sub2, "logpgp95", exog, ["avexpr"], ["logem4"])
    b = res.params["avexpr"]
    se = res.std_errors["avexpr"]
    print(f"  IV {tag:32s} b={b:.3f} (SE {se:.3f})  KP-F={kpf:.2f}  N={int(m_pf._N)}")

The IV coefficient stays between **0.917 and 1.339** — none of these control sets eliminate the institutional-quality effect.

## Robustness 2: geography and climate (Table 6)

In [ ]:
df6 = load_dta(6)
df6 = df6[df6["baseco"] == 1].copy()

temp_cols = [c for c in df6.columns if c.startswith("temp")]
humid_cols = [c for c in df6.columns if c.startswith("humid")]
soil_cols = ["steplow", "deslow", "stepmid", "desmid", "drystep", "drywint",
             "goldm", "iron", "silv", "zinc", "oilres", "landlock"]

table6_specs = [
    ("c1: temp+humid",         temp_cols + humid_cols),
    ("c2: temp+humid+lat",     temp_cols + humid_cols + ["lat_abst"]),
    ("c3: edes1975",           ["edes1975"]),
    ("c4: edes1975+lat",       ["edes1975", "lat_abst"]),
    ("c5: soil/resources",     soil_cols),
    ("c6: soil/resources+lat", soil_cols + ["lat_abst"]),
    ("c7: avelf",              ["avelf"]),
    ("c8: avelf+lat",          ["avelf", "lat_abst"]),
    ("c9: all",                temp_cols + humid_cols + ["edes1975", "avelf"] + soil_cols + ["lat_abst"]),
]

for tag, exog in table6_specs:
    exog_clean = [c for c in exog if c in df6.columns]
    if exog_clean:
        rhs = " + ".join(exog_clean)
        fml = f"logpgp95 ~ {rhs} | avexpr ~ logem4"
    else:
        fml = "logpgp95 ~ 1 | avexpr ~ logem4"
    m_pf, sub2 = iv_with_kpf(df6, fml)
    res, kpf = run_iv2sls(sub2, "logpgp95", exog_clean, ["avexpr"], ["logem4"])
    b = res.params["avexpr"]
    se = res.std_errors["avexpr"]
    print(f"  IV {tag:28s} b={b:.3f} (SE {se:.3f})  KP-F={kpf:.2f}  N={int(m_pf._N)}")

The IV coefficient ranges from **0.713 to 1.358**, bracketing the 0.944 baseline. First-stage F drops below 10 in several columns — geography absorbs first-stage signal.

## Robustness 3: health channels (Table 7)

The tightest challenge to exclusion: does the disease environment directly depress productivity, independent of institutions? Cols 7-9 instrument BOTH `avexpr` AND a health variable using four instruments — `linearmodels` handles multi-endog where `pyfixest` cannot.

In [ ]:
df7 = load_dta(7)
df7 = df7[df7["baseco"] == 1].copy()
df7["other_cont7"] = ((df7["shortnam"] == "AUS") | (df7["shortnam"] == "MLT") |
                     (df7["shortnam"] == "NZL")).astype(int)

# Cols 1-6: just-identified with one health control
table7_specs_simple = [
    ("c1: + malfal94",       ["malfal94"]),
    ("c2: + malfal94 + lat", ["malfal94", "lat_abst"]),
    ("c3: + leb95",          ["leb95"]),
    ("c4: + leb95 + lat",    ["leb95", "lat_abst"]),
    ("c5: + imr95",          ["imr95"]),
    ("c6: + imr95 + lat",    ["imr95", "lat_abst"]),
]

for tag, exog in table7_specs_simple:
    rhs = " + ".join(exog)
    fml = f"logpgp95 ~ {rhs} | avexpr ~ logem4"
    m_pf, sub2 = iv_with_kpf(df7, fml)
    res, kpf = run_iv2sls(sub2, "logpgp95", exog, ["avexpr"], ["logem4"])
    b = res.params["avexpr"]
    se = res.std_errors["avexpr"]
    print(f"  IV {tag:28s} b={b:.3f} (SE {se:.3f})  KP-F={kpf:.2f}  N={int(m_pf._N)}")

In [ ]:
# Cols 7-9: 2 endogenous regressors, 4 instruments => Hansen J meaningful
print("*** Cols 7-9: 2 endog, 4 instruments => Hansen J meaningful ***\n")
table7_specs_multi = [
    ("c7: avexpr + malfal94 endog",  "malfal94"),
    ("c8: avexpr + leb95 endog",     "leb95"),
    ("c9: avexpr + imr95 endog",     "imr95"),
]
inst_set = ["logem4", "latabs", "lt100km", "meantemp"]

for tag, second_endog in table7_specs_multi:
    sub = df7.dropna(subset=["logpgp95", "avexpr", second_endog] + inst_set).copy()
    y_ = sub["logpgp95"].values
    X_exog_ = pd.DataFrame({"const": np.ones(len(sub))}, index=sub.index)
    res = IV2SLS(y_, X_exog_, sub[["avexpr", second_endog]], sub[inst_set]).fit(cov_type="robust")
    b = res.params["avexpr"]
    se = res.std_errors["avexpr"]
    try:
        j_stat = float(res.sargan.stat)
        j_pval = float(res.sargan.pval)
    except (AttributeError, TypeError):
        j_stat, j_pval = np.nan, np.nan
    fs_F = float(res.first_stage.diagnostics.loc["avexpr", "f.stat"])
    print(f"  IV {tag:32s} avexpr b={b:.3f} (SE {se:.3f})  "
          f"Hansen J={j_stat:.2f} (p={j_pval:.3f})  fs-F={fs_F:.2f}  N={len(sub)}")

In [ ]:
# Cols 10-11: yellow fever instrument
print("*** Cols 10-11: yellow-fever instrument ***\n")
table7_specs_yellow = [
    ("c10: yellow",                 [],                                   "yellow"),
    ("c11: yellow + continents",    ["africa", "asia", "other_cont7"],    "yellow"),
]
for tag, exog, inst in table7_specs_yellow:
    rhs = (" + ".join(exog) + " | " if exog else "1 | ")
    fml = f"logpgp95 ~ {rhs}avexpr ~ {inst}"
    m_pf, sub2 = iv_with_kpf(df7, fml)
    res, kpf = run_iv2sls(sub2, "logpgp95", exog, ["avexpr"], [inst])
    b = res.params["avexpr"]
    se = res.std_errors["avexpr"]
    print(f"  IV {tag:32s} b={b:.3f} (SE {se:.3f})  KP-F={kpf:.2f}  N={int(m_pf._N)}")

Health controls shrink the IV coefficient to **0.55-0.69** — the only place where IV approaches OLS. First-stage F collapses to 3.98-5.12 in just-identified specs. Health channels are where fair-minded doubt should remain.

## Overidentification and alternative instruments (Table 8)

AJR use alternative historical-institution variables — `euro1900`, `cons00a`, `democ00a`, `cons1`, `democ1` — to test whether multiple instruments agree on the same causal effect.

In [ ]:
df8 = load_dta(8)
df8 = df8[df8["baseco"] == 1].copy()

# Panels A/B: each alt instrument alone (just-identified)
print("*** Panels A/B: each alternative instrument alone ***\n")
panels_AB = [
    ("a.c1: euro1900",                  [],                "euro1900"),
    ("a.c2: euro1900 + lat",            ["lat_abst"],      "euro1900"),
    ("a.c3: cons00a",                   [],                "cons00a"),
    ("a.c4: cons00a + lat",             ["lat_abst"],      "cons00a"),
    ("a.c5: democ00a",                  [],                "democ00a"),
    ("a.c6: democ00a + lat",            ["lat_abst"],      "democ00a"),
    ("a.c7: cons1 (+ indtime)",         ["indtime"],       "cons1"),
    ("a.c8: cons1 (+ indtime) + lat",   ["indtime", "lat_abst"], "cons1"),
    ("a.c9: democ1 (+ indtime)",        ["indtime"],       "democ1"),
    ("a.c10: democ1 (+ indtime) + lat", ["indtime", "lat_abst"], "democ1"),
]

for tag, exog, inst in panels_AB:
    rhs = (" + ".join(exog) + " | " if exog else "1 | ")
    fml = f"logpgp95 ~ {rhs}avexpr ~ {inst}"
    m_pf, sub2 = iv_with_kpf(df8, fml)
    res, kpf = run_iv2sls(sub2, "logpgp95", exog, ["avexpr"], [inst])
    b = res.params["avexpr"]
    se = res.std_errors["avexpr"]
    print(f"  Panel A/B  {tag:34s} b={b:.3f} (SE {se:.3f})  KP-F={kpf:.2f}  N={int(m_pf._N)}")

In [ ]:
# Panel C: 2 instruments per regression -> Hansen J meaningful
print("*** Panel C: alt instrument + logem4 => Hansen J overid test ***\n")
panel_C = [
    ("c.c1: euro1900 + logem4",                 [],                ["euro1900", "logem4"]),
    ("c.c2: euro1900 + logem4 + lat",           ["lat_abst"],      ["euro1900", "logem4"]),
    ("c.c3: cons00a + logem4",                  [],                ["cons00a", "logem4"]),
    ("c.c4: cons00a + logem4 + lat",            ["lat_abst"],      ["cons00a", "logem4"]),
    ("c.c5: democ00a + logem4",                 [],                ["democ00a", "logem4"]),
    ("c.c6: democ00a + logem4 + lat",           ["lat_abst"],      ["democ00a", "logem4"]),
    ("c.c7: cons1+logem4 (+ indtime)",          ["indtime"],       ["cons1", "logem4"]),
    ("c.c8: cons1+logem4 (+ indtime) + lat",    ["indtime", "lat_abst"], ["cons1", "logem4"]),
    ("c.c9: democ1+logem4 (+ indtime)",         ["indtime"],       ["democ1", "logem4"]),
    ("c.c10: democ1+logem4 (+ indtime) + lat",  ["indtime", "lat_abst"], ["democ1", "logem4"]),
]

for tag, exog, insts in panel_C:
    sub = df8.dropna(subset=["logpgp95", "avexpr"] + exog + insts).copy()
    y_ = sub["logpgp95"].values
    if exog:
        X_exog_ = sub[exog].copy()
        X_exog_["const"] = 1.0
    else:
        X_exog_ = pd.DataFrame({"const": np.ones(len(sub))}, index=sub.index)
    res = IV2SLS(y_, X_exog_, sub[["avexpr"]], sub[insts]).fit(cov_type="robust")
    b = res.params["avexpr"]
    se = res.std_errors["avexpr"]
    try:
        j_stat = float(res.sargan.stat)
        j_pval = float(res.sargan.pval)
    except (AttributeError, TypeError):
        j_stat, j_pval = np.nan, np.nan
    fs_F = float(res.first_stage.diagnostics.loc["avexpr", "f.stat"])
    print(f"  Panel C   {tag:36s} b={b:.3f} (SE {se:.3f})  "
          f"Hansen J={j_stat:.2f} (p={j_pval:.3f})  fs-F={fs_F:.2f}  N={len(sub)}")

In [ ]:
# Panel D: logem4 as exogenous control in second stage
print("*** Panel D: logem4 as exogenous control (relaxes exclusion) ***\n")
panel_D = [
    ("d.c1: euro1900",                  ["logem4"],                    "euro1900"),
    ("d.c2: euro1900 + lat",            ["logem4", "lat_abst"],        "euro1900"),
    ("d.c3: cons00a",                   ["logem4"],                    "cons00a"),
    ("d.c4: cons00a + lat",             ["logem4", "lat_abst"],        "cons00a"),
    ("d.c5: democ00a",                  ["logem4"],                    "democ00a"),
    ("d.c6: democ00a + lat",            ["logem4", "lat_abst"],        "democ00a"),
    ("d.c7: cons1 (+ indtime)",         ["logem4", "indtime"],         "cons1"),
    ("d.c8: cons1 (+ indtime) + lat",   ["logem4", "indtime", "lat_abst"], "cons1"),
    ("d.c9: democ1 (+ indtime)",        ["logem4", "indtime"],         "democ1"),
    ("d.c10: democ1 (+ indtime) + lat", ["logem4", "indtime", "lat_abst"], "democ1"),
]

for tag, exog, inst in panel_D:
    rhs = " + ".join(exog) + " | "
    fml = f"logpgp95 ~ {rhs}avexpr ~ {inst}"
    m_pf, sub2 = iv_with_kpf(df8, fml)
    res, kpf = run_iv2sls(sub2, "logpgp95", exog, ["avexpr"], [inst])
    b = res.params["avexpr"]
    se = res.std_errors["avexpr"]
    print(f"  Panel D   {tag:34s} b={b:.3f} (SE {se:.3f})  KP-F={kpf:.2f}  N={int(m_pf._N)}")

Panel C delivers Hansen J p-values from **0.18 to 0.79** — uniformly failing to reject joint exogeneity. But Albouy (2012) warns that ~36% of mortality observations are imputed or shared, limiting the power of this test.

## Visual summary: OLS vs IV across specifications (Figure 3)

In [ ]:
def iv_beta_ci(df_in, exog, endog, inst):
    sub = df_in.dropna(subset=["logpgp95"] + exog + endog + inst).copy()
    if exog:
        X_exog_ = sub[exog].copy()
        X_exog_["const"] = 1.0
    else:
        X_exog_ = pd.DataFrame({"const": np.ones(len(sub))}, index=sub.index)
    res = IV2SLS(sub["logpgp95"].values, X_exog_, sub[endog], sub[inst]).fit(cov_type="robust")
    b = res.params["avexpr"]
    ci_lo = res.conf_int().loc["avexpr", "lower"]
    ci_hi = res.conf_int().loc["avexpr", "upper"]
    return b, ci_lo, ci_hi

def ols_beta_ci(df_in, fml):
    sub = df_in.dropna(subset=[c for c in vars_in(fml) if c in df_in.columns])
    m = pf.feols(fml, data=sub, vcov="HC1")
    b, se = coef_se(m, "avexpr")
    return b, b - 1.96 * se, b + 1.96 * se

labels, betas, ci_los, ci_his, colors = [], [], [], [], []

# OLS Tab 2 base sample
b, lo, hi = ols_beta_ci(df2[df2["baseco"] == 1], "logpgp95 ~ avexpr")
labels.append("OLS\n(Tab 2)"); betas.append(b); ci_los.append(lo); ci_his.append(hi); colors.append(WARM_ORANGE)

# IV Tab 4 base
b, lo, hi = iv_beta_ci(df4, [], ["avexpr"], ["logem4"])
labels.append("IV: settler\nmortality\n(Tab 4)"); betas.append(b); ci_los.append(lo); ci_his.append(hi); colors.append(STEEL_BLUE)

# IV Tab 5 colonial
b, lo, hi = iv_beta_ci(df5, ["f_brit", "f_french"], ["avexpr"], ["logem4"])
labels.append("IV + colonial\ncontrols\n(Tab 5)"); betas.append(b); ci_los.append(lo); ci_his.append(hi); colors.append(STEEL_BLUE)

# IV Tab 6 geography
exog6 = [c for c in df6.columns if c.startswith(("temp", "humid"))]
b, lo, hi = iv_beta_ci(df6, exog6, ["avexpr"], ["logem4"])
labels.append("IV + geography\ncontrols\n(Tab 6)"); betas.append(b); ci_los.append(lo); ci_his.append(hi); colors.append(STEEL_BLUE)

# IV Tab 7 malaria
b, lo, hi = iv_beta_ci(df7, ["malfal94"], ["avexpr"], ["logem4"])
labels.append("IV + malaria\ncontrol\n(Tab 7)"); betas.append(b); ci_los.append(lo); ci_his.append(hi); colors.append(STEEL_BLUE)

# IV Tab 8 alt instrument
b, lo, hi = iv_beta_ci(df8, [], ["avexpr"], ["euro1900"])
labels.append("IV: alt inst\neuro1900\n(Tab 8)"); betas.append(b); ci_los.append(lo); ci_his.append(hi); colors.append(TEAL)

fig3, ax3 = plt.subplots(figsize=(10, 6))
ypos = np.arange(len(labels))[::-1]
for yi, (b, lo, hi, col) in enumerate(zip(betas, ci_los, ci_his, colors)):
    yp = ypos[yi]
    ax3.errorbar(b, yp, xerr=[[b - lo], [hi - b]], fmt="o", color=col,
                 ecolor=col, elinewidth=1.5, capsize=4, markersize=9, alpha=0.95)
ax3.axvline(0, color=LIGHT_TEXT, linestyle="--", linewidth=0.8, alpha=0.6)
ax3.set_yticks(ypos)
ax3.set_yticklabels(labels, color=LIGHT_TEXT, fontsize=9)
ax3.set_xlabel("Coefficient on avexpr (institutions)")
ax3.set_title("Effect of institutions on log GDP: OLS vs IV", fontsize=13, color=WHITE_TEXT, pad=22)
ax3.text(0.5, 1.02, "Six representative specs, 95% CI, AJR (2001) base sample",
         transform=ax3.transAxes, ha="center", fontsize=10, color=LIGHT_TEXT)
plt.tight_layout()
plt.show()

## Summary and limitations

- **2SLS = 0.944** vs OLS = 0.522 — IV is **81% larger**, consistent with measurement-error attenuation dominating in OLS.
- **Wu-Hausman** rejects OLS consistency (F = 24.22, p < 0.0001).
- **First-stage F = 16.85** — borderline-strong, just above Stock-Yogo 10% threshold (16.38).
- **Hansen J** p-values 0.18-0.79 — fails to reject joint exogeneity across 5 instrument pairs.
- The 0.944 is a **LATE** for compliers, not a population ATE.
- Albouy (2012): ~36% of mortality data are imputed/shared — Hansen J has limited power against shared imputation bias.
- Health channels (Table 7) are the place where honest doubt should remain.

**References:**
1. Acemoglu, Johnson, Robinson (2001). AER 91(5): 1369-1401.
2. Albouy (2012). AER 102(6): 3059-3076.
3. Imbens and Angrist (1994). Econometrica 62(2): 467-475.
4. Stock and Yogo (2005). In Andrews-Stock Festschrift.